# DACON 236753 — Colab 제출·검증 러너

이 노트북은 제공된 `dacon236753_colab_bundle.zip`을 풀고, 공식 베이스라인을 GPU에서 smoke 실행한 뒤 **제출 규격의 `submit.zip`**을 생성합니다.

- 현재 번들은 공식 baseline과 예제 fixture를 포함합니다.
- smoke 점수는 예제 학습 데이터에 대한 회귀 점검용이며 Public 점수 추정용이 아닙니다.
- `submit.zip`은 최상위 폴더 없이 `model/`, 독립형 `inference.py`, `requirements.txt`만 포함합니다.

In [ ]:
# Colab 메뉴: 런타임 > 런타임 유형 변경 > GPU 를 먼저 선택하세요.
!nvidia-smi
!python --version

In [ ]:
# 권장: 500MB 안팎의 bundle은 Google Drive에 올린 뒤 경로를 지정하세요.
USE_GOOGLE_DRIVE = True
DRIVE_ROOT = '/content/drive/MyDrive/블랙박스 영상 기반 사고 분석'
DRIVE_BUNDLE_PATH = f'{DRIVE_ROOT}/dacon236753_colab_bundle.zip'  # 최초 1회용 base
DRIVE_PATCH_PATH = f'{DRIVE_ROOT}/dacon236753_patch.zip'          # 이후 수정용

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BUNDLE_PATH = DRIVE_BUNDLE_PATH
    PATCH_PATH = DRIVE_PATCH_PATH
else:
    from google.colab import files
    uploaded = files.upload()
    BUNDLE_PATH = next(iter(uploaded))
    uploaded_patch = files.upload()
    PATCH_PATH = next(iter(uploaded_patch))
print('Base:', BUNDLE_PATH)
print('Patch:', PATCH_PATH)

In [ ]:
# 새 작업 경로를 만들고 bundle을 풉니다.
!rm -rf /content/dacon236753
!mkdir -p /content/dacon236753
!unzip -q "$BUNDLE_PATH" -d /content/dacon236753
!unzip -q -o "$PATCH_PATH" -d /content/dacon236753
%cd /content/dacon236753
!find . -maxdepth 2 -type f | sort | sed -n '1,80p'

In [ ]:
# 평가 서버와 호환되는 Torch/TorchVision 조합을 설치합니다.
# 이 셀을 실행한 뒤 아래 import 셀에서 문제가 생기면 런타임을 한 번 재시작한 후 그 셀부터 다시 실행하세요.
!python -m pip install -q --upgrade --no-cache-dir --index-url https://download.pytorch.org/whl/cu128 torch==2.8.0 torchvision==0.23.0
# cv2, pandas, Pillow은 Colab 기본 설치본을 사용합니다. 고정 버전 재설치는
# Python 버전에 따라 소스 빌드로 전환되어 수십 분 멈춘 것처럼 보일 수 있으므로 하지 않습니다.

In [ ]:
# GPU와 핵심 라이브러리 확인. 실패하면 이후 제출을 진행하지 마세요.
import cv2, pandas as pd, torch, torchvision
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('cv2:', cv2.__version__, 'pandas:', pd.__version__)
!python tools/preflight.py

In [ ]:
# 공식 예제를 실제 평가 입력과 같은 폴더 구조로 변환합니다.
!python tools/prepare_baseline_smoke.py --baseline-data fixtures/baseline_data --out artifacts/baseline_smoke --overwrite

In [ ]:
# 원본 제출 코드로 3개 Stage를 모두 GPU 추론합니다. 수 분이 걸릴 수 있습니다.
!python tools/run_local_inference.py --input-root artifacts/baseline_smoke/input --model-dir model --out artifacts/baseline_smoke/predictions

In [ ]:
# 범주·중복·원본 프레임 범위·Stage 3 sample 누락을 먼저 차단합니다.
!python tools/check_predictions.py --predictions-dir artifacts/baseline_smoke/predictions --stage1-videos artifacts/baseline_smoke/input/stage1/videos --stage2-images artifacts/baseline_smoke/input/stage2/images --stage3-expected artifacts/baseline_smoke/input/stage3_expected.csv
!python tools/local_validate.py --labels-dir artifacts/baseline_smoke/labels --predictions-dir artifacts/baseline_smoke/predictions --stage2-frame-time-map artifacts/baseline_smoke/labels/stage2_frame_time.csv --json-out artifacts/baseline_smoke/local_metrics.json

In [ ]:
# 정확한 제출 archive를 생성하고, ZIP 구조·문법·import를 검사합니다.
!python tools/build_submit.py --project-root . --output artifacts/submit/submit.zip
!python tools/validate_submit_package.py --zip artifacts/submit/submit.zip --import-check
!unzip -l artifacts/submit/submit.zip

In [ ]:
# 2차 실행: 방금 만든 ZIP을 풀어, ZIP 내부 코드와 가중치로 다시 smoke 추론합니다.
!rm -rf artifacts/submit_check artifacts/baseline_smoke/predictions_from_zip
!unzip -q artifacts/submit/submit.zip -d artifacts/submit_check
!python tools/run_local_inference.py --input-root artifacts/baseline_smoke/input --model-dir artifacts/submit_check/model --inference artifacts/submit_check/inference.py --out artifacts/baseline_smoke/predictions_from_zip
!python tools/check_predictions.py --predictions-dir artifacts/baseline_smoke/predictions_from_zip --stage1-videos artifacts/baseline_smoke/input/stage1/videos --stage2-images artifacts/baseline_smoke/input/stage2/images --stage3-expected artifacts/baseline_smoke/input/stage3_expected.csv
!python tools/local_validate.py --labels-dir artifacts/baseline_smoke/labels --predictions-dir artifacts/baseline_smoke/predictions_from_zip --stage2-frame-time-map artifacts/baseline_smoke/labels/stage2_frame_time.csv --json-out artifacts/baseline_smoke/local_metrics_from_zip.json

In [ ]:
# 모든 PASS를 확인한 경우에만 다운로드해 DACON 제출 탭에 업로드하세요.
from google.colab import files
files.download('artifacts/submit/submit.zip')